# Responsive Routing Event Explorer

Visualises individual **diff-sat** events: 15-second windows where the two co-located dishes  
connect to **different** satellites. For each event chunk the notebook shows:

- **Left panel** — azimuth/elevation sky-plot with SE obstruction boundaries and all visible satellite positions
- **Right panel** — second-granularity RTT (control=green, test=red) + loss (twin y-axis) + first-hop RTT annotation
- **Obstruction map** — the raw 123×123 px PNG written by the Starlink app for that window

Adjust `start_date` / `end_date` and `skip_first_n` / `total_plots_to_show` to explore different days.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'process'))

# Processing scripts (IRTT, TLE, pipeline): see process/
from data_loading import *
from satellite_matching import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from tqdm import tqdm


## Load Data

In [ ]:
est = timezone('US/Eastern')

# Adjust date range to explore a different period
start_date_str = "2025-01-01"
end_date_str   = "2025-01-03"

start_date = datetime(2025, 1, 1, 0, 0, 0, 0, est)
end_date   = datetime(2025, 1, 3, 23, 59, 59, 0, est)

rst_obsmap_dict = get_rst_obsmap_dict(start_date, end_date)
sat_match_data  = read_sat_match_data(start_date_str, end_date_str)
all_sat_pc_df   = read_all_sat_pc(start_date_str, end_date_str)
rtt_data_df     = read_rtt_data(start_date_str, end_date_str)
tr_fh_data_df   = get_traceroute_fh_latency(start_date_str, end_date_str)
loss_data       = get_loss_data(start_date_str, end_date_str)

In [ ]:
# Build diff-sat event chunks
all_sat_data = pd.merge(sat_match_data, all_sat_pc_df, on="date", how="left")
all_sat_data = all_sat_data.sort_values("date").reset_index(drop=True)

diff_indices = all_sat_data.index[all_sat_data["pi1_sat"] != all_sat_data["pi2_sat"]]
extended = set(diff_indices)
for idx in diff_indices:
    if idx > 0: extended.add(idx - 1)
    if idx < len(all_sat_data) - 1: extended.add(idx + 1)

diff_sat_data = all_sat_data.loc[sorted(extended)].reset_index(drop=True)

chunks, current_chunk = [], []
for i, row in diff_sat_data.iterrows():
    if not current_chunk:
        current_chunk.append(row['date'])
    else:
        if row['pi1_sat'] != row['pi2_sat']:
            current_chunk.append(row['date'])
        else:
            current_chunk.append(row['date'])
            chunks.append(current_chunk)
            current_chunk = []

filtered_chunks = []
for chunk in chunks:
    contiguous = all(
        pd.to_datetime(chunk[i]) + pd.Timedelta(seconds=15) == pd.to_datetime(chunk[i+1])
        for i in range(len(chunk) - 1)
    )
    if contiguous:
        filtered_chunks.append(chunk)

print(f"Found {len(filtered_chunks)} contiguous diff-sat event chunks")

## Event Visualisation

Each event chunk shows one row of subplots per 15-second window.  
`skip_first_n` skips the first N chunks; `total_plots_to_show` caps how many are rendered.

In [ ]:
AOE    = [35, 60]
ELE_NE = [3.4, 56.6]
ELE_SE = [93.4, 146.6]

# --- controls ---
skip_first_n       = 20   # skip this many chunks before plotting
total_plots_to_show = 30   # stop after this many plots

rtt_data_df_ms = rtt_data_df.copy()

for time_chunks in filtered_chunks:
    skip = False
    for date in time_chunks:
        start_time = date - pd.Timedelta(seconds=16)
        end_time   = date
        rtt_chunk  = rtt_data_df_ms[
            (rtt_data_df_ms['date'] > start_time) & (rtt_data_df_ms['date'] < end_time)
        ]
        if rtt_chunk.empty:
            skip = True
        if rtt_chunk['rtt_pi1'].isnull().all() or rtt_chunk['rtt_pi2'].isnull().all():
            skip = True

    if skip:
        continue

    if skip_first_n > 0:
        skip_first_n -= 1
        continue

    total_plots_to_show -= 1
    fig, axs = plt.subplots(len(time_chunks), 2)
    fig.set_size_inches(15, 3 * len(time_chunks))

    # --- right column: RTT + loss ---
    for date in time_chunks:
        start_time = date - pd.Timedelta(seconds=16)
        end_time   = date
        rtt_chunk  = rtt_data_df_ms[
            (rtt_data_df_ms['date'] > start_time) & (rtt_data_df_ms['date'] < end_time)
        ]
        idx = time_chunks.index(date)
        ax  = axs[idx][1]
        ax.set_ylim(0, 500)
        ax.set_xlabel("Timestamp")
        ax.set_ylabel("RTT")
        ax.set_title(f"date {date}")
        ax.scatter(rtt_chunk['date'], rtt_chunk['rtt_pi1'], color='green', label="rtt_pi1", s=2)
        ax.scatter(rtt_chunk['date'], rtt_chunk['rtt_pi2'], color='red',   label="rtt_pi2", s=3)
        ax.xaxis.set_major_locator(mdates.SecondLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%S'))
        ax.xaxis.set_minor_locator(mdates.SecondLocator(interval=5))
        ax.grid()

        # First-hop RTT annotation
        fh_chunk = tr_fh_data_df[
            (tr_fh_data_df['date'] > start_time) & (tr_fh_data_df['date'] < end_time)
        ]
        if not fh_chunk.empty:
            x = rtt_chunk['date'].iloc[-1]
            if not fh_chunk['rtt_pi1'].isnull().any():
                ax.text(x - pd.Timedelta(seconds=5), 470,
                        f"fh_rtt_pi1: {fh_chunk['rtt_pi1'].values[0]}", fontsize=12, color='green')
            if not fh_chunk['rtt_pi2'].isnull().any():
                ax.text(x - pd.Timedelta(seconds=5), 440,
                        f"fh_rtt_pi2: {fh_chunk['rtt_pi2'].values[0]}", fontsize=12, color='red')

        # Loss on twin y-axis
        ax2 = ax.twinx()
        loss_chunk = loss_data[
            (loss_data['date'] > start_time) & (loss_data['date'] < end_time)
        ]
        if not loss_chunk.empty:
            ax2.scatter(loss_chunk['date'], loss_chunk['pi1_loss'], color='black', label="pi1_loss", s=10)
            ax2.scatter(loss_chunk['date'], loss_chunk['pi2_loss'], color='blue',  label="pi2_loss", s=10)
            ax2.set_ylim(0, 6)

    # --- left column: sky az/el plot ---
    for date in time_chunks:
        idx = time_chunks.index(date)
        ax  = axs[idx][0]
        ax.grid()
        ax.set_xlabel("Azimuth")
        ax.set_ylabel("Elevation")
        ax.set_title(f"date {date}")
        ax.set_xlim(0, 360)
        ax.set_ylim(0, 90)
        ax.axhline(y=AOE[0], color='r', linestyle='dashed', label="AOE")
        ax.axhline(y=AOE[1], color='r', linestyle='dashed')
        ax.axvline(x=ELE_SE[0], color='g', linestyle='dashed', label="ELE_SE")
        ax.axvline(x=ELE_SE[1], color='g', linestyle='dashed')
        ax.set_yticks(np.arange(0, 90, 10))

        row    = diff_sat_data.loc[diff_sat_data['date'] == date]
        pi1_az = row["pi1_first_az"].values[0]
        pi1_el = row["pi1_first_ele"].values[0]
        pi2_az = row["pi2_first_az"].values[0]
        pi2_el = row["pi2_first_ele"].values[0]
        pi1_sat = row["pi1_sat"].values[0]
        pi2_sat = row["pi2_sat"].values[0]

        for sat in row['all_sat_first_pc'].to_list()[0]:
            sat_vals = list(sat.values())
            ax.scatter(sat_vals[0][1], sat_vals[0][0], color="blue", s=10)

        ax.scatter(pi2_az, pi2_el, color="red",   s=30)
        ax.scatter(pi1_az, pi1_el, color="green", s=30)
        ax.text(0,  0, f"pi1_sat: {pi1_sat}", fontsize=12, color='green')
        ax.text(0, 10, f"pi2_sat: {pi2_sat}", fontsize=12, color='red')

    plt.show()
    show_obs_image(time_chunks[0], rst_obsmap_dict)

    if total_plots_to_show <= 0:
        break